In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression as LR 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split as TTS

# --- ORIGINAL LOCAL PATHS ---
train_data = pd.read_csv("train.csv")
test_data = pd.read_csv("test.csv")

# Original feature list (with Fare)
features = ["Pclass", "Age", "SibSp", "Sex", "Embarked", "Fare"]

In [ ]:
# Clean missing values
median_age = train_data['Age'].median()
train_data['Age'] = train_data['Age'].fillna(median_age)
test_data['Age'] = test_data['Age'].fillna(median_age)

most_common_port = train_data['Embarked'].mode()[0]
train_data['Embarked'] = train_data['Embarked'].fillna(most_common_port) 
test_data['Embarked'] = test_data['Embarked'].fillna(most_common_port)

median_fare = train_data['Fare'].median()
train_data['Fare'] = train_data['Fare'].fillna(median_fare)
test_data['Fare'] = test_data['Fare'].fillna(median_fare)

In [ ]:
# One-Hot Encoding
all_data = pd.concat([train_data[features], test_data[features]], axis=0)
all_data_encoded = pd.get_dummies(all_data, columns=["Sex", "Embarked"])

X_train_full = all_data_encoded.iloc[:len(train_data)]  
y_train_full = train_data["Survived"]
X_test = all_data_encoded.iloc[len(train_data):]   

# Split data into train and validation sets
X_train_raw, X_val_raw, y_train, y_val = TTS(X_train_full, y_train_full, test_size=0.2, random_state=42)

# Scale features (mandatory for Logistic Regression)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test)

# Initialize Logistic Regression
model = LR(class_weight='balanced', max_iter=1000, C=1.0, random_state=42)

# Train and evaluate
model.fit(X_train, y_train)
val_preds = model.predict(X_val)

In [ ]:
precision = precision_score(y_val, val_preds)
recall = recall_score(y_val, val_preds)
f1 = f1_score(y_val, val_preds)

print("--- VALIDATION DATA SCORES (Logistic Regression) ---")
print("Precision:", precision)
print("Recall:", recall)
print("F1 score:", f1)